# 1. What you'll learn

        - separate transition persistence from emission evidence
- implement Viterbi in log space
- erase a real short regime with an over-sticky transition prior

        **The one question this notebook answers:** How can an unseen regime be decoded without letting every noisy observation flip the state?

# 2. Setup

This lesson keeps hidden markov models and viterbi small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from scipy.stats import norm
from sklearn.metrics import accuracy_score, confusion_matrix

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A seeded time series switches between calm σ=0.5 and volatile σ=2 regimes. We know hidden truth only because the sequence is simulated.

**Small example:** For [0.1,0.3,2.7,-2.2,0.2], emissions favour volatility twice; Viterbi also charges for entering and leaving, encouraging one coherent spell.

In [ ]:
rng=np.random.default_rng(SEED);n=450;A=np.array([[.96,.04],[.10,.90]]);sigmas=np.array([.5,2.0]);states=np.zeros(n,dtype=int)
for t in range(1,n): states[t]=rng.choice(2,p=A[states[t-1]])
obs=rng.normal(0,sigmas[states]);frame=pd.DataFrame({"time":np.arange(n),"return":obs,"regime":states})
print("Shape:",frame.shape,"balance:",frame.regime.value_counts(normalize=True).round(3).to_dict(),"transitions:",np.sum(states[1:]!=states[:-1]))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
runs=[];start=0
for i in range(1,n+1):
    if i==n or states[i]!=states[start]: runs.append((states[start],i-start));start=i
x=np.linspace(-6,6,300);fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(obs,lw=.8);axes[0].fill_between(np.arange(n),-6,6,where=states==1,alpha=.15,color="#d38b45");axes[0].set_title("Orange = volatile truth")
axes[1].hist([[r[1] for r in runs if r[0]==0],[r[1] for r in runs if r[0]==1]],label=["calm","volatile"]);axes[1].set_title("Persistent run lengths");axes[1].legend()
axes[2].plot(x,norm.pdf(x,0,.5),label="calm");axes[2].plot(x,norm.pdf(x,0,2),label="volatile");axes[2].set_title("Overlapping emissions");axes[2].legend();plt.tight_layout();plt.show()

**Takeaway:** Large movements cluster inside persistent volatile spans.

**Takeaway:** Runs are longer than independent state labels would create.

**Takeaway:** Small observations are plausible under either regime, so local evidence is ambiguous.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
cut=300;train=frame.iloc[:cut];test=frame.iloc[cut:];scales=train.groupby("regime")["return"].std().to_numpy();counts=np.ones((2,2))
for a,b in zip(train.regime[:-1],train.regime[1:]): counts[a,b]+=1
transition=counts/counts.sum(1,keepdims=True);prior=np.bincount(train.regime,minlength=2)/len(train);print("Scales:",scales.round(3),"transition:\n",transition.round(3))

# 6. Train

        Training turns the assumptions behind hidden markov models and viterbi into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Initialize state scores from priors and the first emission.
2. Add every previous-to-current transition score.
3. Keep the best predecessor for each current state.
4. Backtrack from the best final state.

In [ ]:
def viterbi(values,A,scales,prior):
 T=len(values);score=np.empty((T,2));back=np.zeros((T,2),int);score[0]=np.log(prior)+norm.logpdf(values[0],0,scales)
 for t in range(1,T):
  for s in range(2):
   choices=score[t-1]+np.log(A[:,s]);back[t,s]=np.argmax(choices);score[t,s]=choices[back[t,s]]+norm.logpdf(values[t],0,scales[s])
 path=np.zeros(T,int);path[-1]=np.argmax(score[-1])
 for t in range(T-2,-1,-1): path[t]=back[t+1,path[t+1]]
 return path
decoded=viterbi(test["return"].to_numpy(),transition,scales,prior);print("Decoded counts:",np.bincount(decoded,minlength=2))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
truth=test.regime.to_numpy();emission=np.argmax(np.c_[norm.logpdf(test["return"],0,scales[0]),norm.logpdf(test["return"],0,scales[1])],1);print(f"Viterbi={accuracy_score(truth,decoded):.3f}; emission-only={accuracy_score(truth,emission):.3f}")
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(test.time,test["return"],lw=.8);axes[0].fill_between(test.time,-6,6,where=truth==1,alpha=.18);axes[0].set_title("True volatile spans")
axes[1].plot(test.time,test["return"],lw=.8);axes[1].fill_between(test.time,-6,6,where=decoded==1,alpha=.18,color="#d38b45");axes[1].set_title("Viterbi spans")
sns.heatmap(confusion_matrix(truth,decoded),annot=True,fmt="d",cbar=False,ax=axes[2]);axes[2].set_title("State confusion");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: The Markov transition model can be too sticky, while overlapping emissions make boundaries uncertain. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
sticky=np.array([[.9995,.0005],[.0005,.9995]]);bad=viterbi(test["return"].to_numpy(),sticky,scales,prior)
print(f"Reasonable accuracy={accuracy_score(truth,decoded):.3f}; over-sticky={accuracy_score(truth,bad):.3f}; volatile points erased={np.sum((truth==1)&(bad==0))}")

**Use this when…** a sequence arises from a few persistent latent regimes with different observation patterns.

        **Don't use this when…** durations are not memoryless, transitions depend on rich context, or regimes drift.

        ## Try this

        1. Move the emission scales closer and watch transition assumptions dominate.
2. Use a uniform transition matrix and count fragmented runs.
3. Add a third high-mean state and generalize loops to K.